In [ ]:
!pip install -q torch==2.4.1 triton==3.0.0

In [ ]:
import torch
import triton
import triton.language as tl

## Preprocessing

In [7]:
def pack_blocks(mat: torch.Tensor, block_size: int):
    """
    Pack an EW N:M sparse matrix along the reduction dimension (K).

    Assumes mat is 2D with shape (M, K) and sparsity is enforced *per
    contiguous block of `block_size` elements along K for each row*
    (EW-N:M, e.g., 2 nonzeros in every 4 weights along K). This matches
    the paper's definition and SparTA's packing expectation.

    Returns:
      vals:         concatenated non-zeros
      idx_in_block: their positions within each block (0..block_size-1)
      block_ptr:    prefix-sum into vals per block (len = n_blocks + 1)
      block_coords: (row, k_start) for each block to locate it in mat
    """
    if mat.dim() != 2:
        raise ValueError("pack_blocks expects a 2D tensor (M, K)")

    M, K = mat.shape
    vals_list = []
    idx_list = []
    block_ptr = [0]
    block_coords = []

    for row in range(M):
        row_vec = mat[row]
        for k_start in range(0, K, block_size):
            block = row_vec[k_start:k_start + block_size]
            nz_idx = (block != 0).nonzero(as_tuple=False).squeeze(-1)
            # append block info
            vals_list.append(block[nz_idx])
            idx_list.append(nz_idx)
            block_ptr.append(block_ptr[-1] + nz_idx.numel())
            block_coords.append((row, k_start))

    # concatenate (includes empty blocks so block_ptr stays aligned)
    vals = torch.cat(vals_list, dim=0) if vals_list else mat.new_empty((0,))
    idx_in_block = torch.cat(idx_list, dim=0) if idx_list else torch.empty(0, dtype=torch.long, device=mat.device)
    block_ptr = torch.tensor(block_ptr, dtype=torch.long, device=mat.device)
    block_coords = torch.tensor(block_coords, dtype=torch.long, device=mat.device) if block_coords else torch.empty((0, 2), dtype=torch.long, device=mat.device)

    return vals, idx_in_block, block_ptr, block_coords

# Call this after defining your matrix, e.g. see the next cell.

In [8]:
# Example: 2:4 EW-N:M sparse matrix along reduction dim K (block_size=4)
# Each block of 4 contiguous entries *per row* has exactly 2 non-zeros.
A = torch.tensor([
    # blocks: [0..3] , [4..7]
    [5, 0, 7, 0,   0, 2, 0, 3],
    [0, 4, 0, 6,   8, 0, 1, 0],
    [9, 0, 0, 10,  0, 11, 12, 0],
    [0, 13, 14, 0,  15, 0, 0, 16],
], dtype=torch.float32)

block_size = 4  # matches the 2:4 pattern used in SparTA-style packing
vals, idx_in_block, block_ptr, block_coords = pack_blocks(A, block_size=block_size)
print("vals:", vals)                         # nonzeros in block order
print("idx_in_block:", idx_in_block)           # positions 0..3 within each block
print("block_ptr:", block_ptr)                 # block b -> vals[block_ptr[b]:block_ptr[b+1]]
print("block_coords (row, k_start):", block_coords)

# quick sanity check for EW 2:4 sparsity (along K)
block_counts = block_ptr[1:] - block_ptr[:-1]
assert (block_counts == 2).all(), "Matrix violates EW 2:4 sparsity along K"
print("nonzeros per block:", block_counts)

vals: tensor([5., 1., 2., 3., 4.])
idx_in_block: tensor([1, 0, 3, 2, 0])
block_ptr: tensor([0, 1, 3, 4, 5])
